In [1]:
from datasets import load_dataset

alpaca_dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Len of Alpaca dataset : {len(alpaca_dataset)}")

/opt/conda/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Len of Alpaca dataset : 52002


In [2]:
data_10k = alpaca_dataset.select(range(10000))
data_10k

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 10000
})

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 15899.14it/s]


In [ ]:
def build_prompt_template(instruction, inp=""): 
    user = instruction if not inp else f"{instruction}\n\n{inp}"
    return f"### Instruction:\n{user}\n\n### Response:\n"

lengths = []
more_than_512 = 0 # count of samples exceeding this length
more_than_512_lengths = []

response_tokens_retained_pct = [] # After cutting the sequence to 512 tokens, how many answer tokens are left for the model to learn from?
zero_response_ids = 0

for idx in range(10000): 
    sample = data_10k[idx]

    instruction = sample["instruction"]
    output = sample["output"]
    inp = sample["input"]

    prompt_text = build_prompt_template(instruction, inp)
    response_text = output
    
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    input_ids = prompt_ids + response_ids 

    length = len(input_ids)
    lengths.append(length)

    if length > 512: 
        more_than_512 += 1
        more_than_512_lengths.append(length)
        print(f"Case {idx} :: len of input ids : {len(input_ids)} :: len of prompt ids : {len(prompt_ids)} :: len of response ids : {len(response_ids)}")

    space_for_response = 512 - len(prompt_ids)
    if space_for_response >= len(response_ids) :  # Enough space
        response_tokens_retained_pct.append(100) 

    elif space_for_response < 0 : # No space
        response_tokens_retained_pct.append(0)
        zero_response_ids += 1

    else: # some space case 
        response_tokens_retained_pct.append((space_for_response/ len(response_ids))* 100)


print(f"Percentage of samples with more than 512 tokens : {(more_than_512/10000) * 100}")
print(f"Average length of input ids : {sum(lengths)/len(lengths)}")
print(f"More than 512 ids lengths : {more_than_512_lengths}")
print(f"Average response tokens retained pct : {sum(response_tokens_retained_pct) / len(response_tokens_retained_pct)}")
print(f"Zero response is present in 512 length : {zero_response_ids}")

Case 3559 :: len of input ids : 623 :: len of prompt ids : 256 :: len of response ids : 367
Case 6285 :: len of input ids : 1090 :: len of prompt ids : 27 :: len of response ids : 1063
Case 6676 :: len of input ids : 524 :: len of prompt ids : 18 :: len of response ids : 506
Case 9066 :: len of input ids : 552 :: len of prompt ids : 25 :: len of response ids : 527
Percentage of samples with more than 512 tokens : 0.04
Average length of input ids : 79.2198
More than 512 ids lengths : [623, 1090, 524, 552]
Average response tokens retained pct : 99.99054186820216
Zero response is present in 512 length : 0


In [6]:
tokenizer.pad_token, tokenizer.eos_token, tokenizer.pad_token_id, tokenizer.eos_token_id

('<|endoftext|>', '<|endoftext|>', 151643, 151643)

In [8]:
from collections import Counter

n = len(data_10k)

empty_outputs = []
malformed = []
duplicate_instructions = Counter()
duplicate_examples = Counter()
short_outputs = []

for i, sample in enumerate(data_10k):
    instruction = sample.get("instruction")
    inp = sample.get("input")
    output = sample.get("output")

    # 1. Empty or whitespace-only outputs
    if not isinstance(output, str) or not output.strip():
        empty_outputs.append(i)
    elif len(output.strip()) < 10:
        short_outputs.append(i)

    # 2. Malformed examples
    if (
        not isinstance(instruction, str) or not instruction.strip()
        or not isinstance(inp, str)
        or not isinstance(output, str)
        or not output.strip()
    ):
        malformed.append(i)

    # 3. Duplicate instructions and complete examples
    if isinstance(instruction, str):
        duplicate_instructions[instruction.strip()] += 1

    if all(isinstance(x, str) for x in [instruction, inp, output]):
        key = (instruction.strip(), inp.strip(), output.strip())
        duplicate_examples[key] += 1

duplicate_instruction_groups = sum(
    count > 1 for count in duplicate_instructions.values()
)
duplicate_example_groups = sum(
    count > 1 for count in duplicate_examples.values()
)
duplicate_example_rows = sum(
    count - 1 for count in duplicate_examples.values() if count > 1
)

print(f"Total samples: {n}")
print(f"Empty/invalid outputs: {len(empty_outputs)}")
print(f"Malformed samples: {len(malformed)}")
print(f"Outputs shorter than 10 characters: {len(short_outputs)}")
print(f"Duplicate instruction groups: {duplicate_instruction_groups}")
print(f"Duplicate complete-example groups: {duplicate_example_groups}")
print(f"Extra rows from exact duplicates: {duplicate_example_rows}")

print("Empty output indices:", empty_outputs[:20])
print("Malformed indices:", malformed[:20])
print("Short output indices:", short_outputs[:20])



Total samples: 10000
Empty/invalid outputs: 6
Malformed samples: 6
Outputs shorter than 10 characters: 537
Duplicate instruction groups: 0
Duplicate complete-example groups: 0
Extra rows from exact duplicates: 0
Empty output indices: [284, 1223, 1347, 1736, 7723, 9038]
Malformed indices: [284, 1223, 1347, 1736, 7723, 9038]
Short output indices: [5, 81, 99, 103, 122, 154, 155, 157, 180, 186, 209, 210, 211, 218, 226, 230, 244, 266, 270, 287]


In [9]:
print("Empty outputs")
for i in empty_outputs:
    print(f"\nIndex: {i}")
    print(data_10k[i])


Index: 284
{'instruction': 'Standardize the following date to the ISO 8601 format.', 'input': 'July 26th, 2021', 'output': '', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nStandardize the following date to the ISO 8601 format.\n\n### Input:\nJuly 26th, 2021\n\n### Response:\n'}

Index: 1223
{'instruction': 'Create an illustration for the following text.', 'input': 'Gun violence in the United States can result in death, injury, destruction of property, and a wide range of other difficulties.', 'output': '', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate an illustration for the following text.\n\n### Input:\nGun violence in the United States can result in death, injury, destruction of property, and a 

In [10]:
print("Malformed samples")
for i in malformed:
    print(f"\nIndex: {i}")
    print(data_10k[i])

Malformed samples

Index: 284
{'instruction': 'Standardize the following date to the ISO 8601 format.', 'input': 'July 26th, 2021', 'output': '', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nStandardize the following date to the ISO 8601 format.\n\n### Input:\nJuly 26th, 2021\n\n### Response:\n'}

Index: 1223
{'instruction': 'Create an illustration for the following text.', 'input': 'Gun violence in the United States can result in death, injury, destruction of property, and a wide range of other difficulties.', 'output': '', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate an illustration for the following text.\n\n### Input:\nGun violence in the United States can result in death, injury, destruction o

In [11]:
print("Short outpus : 30 ")
for i in short_outputs[:30]:
    print(f"\nIndex: {i}")
    print(data_10k[i])

Short outpus : 30 

Index: 5
{'instruction': 'Identify the odd one out.', 'input': 'Twitter, Instagram, Telegram', 'output': 'Telegram', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nIdentify the odd one out.\n\n### Input:\nTwitter, Instagram, Telegram\n\n### Response:\nTelegram'}

Index: 81
{'instruction': 'Classify the following incident as a breach of protocol. Output 1 for breach, and 0 for no breach.', 'input': 'Using a school laptop for personal use', 'output': '1', 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nClassify the following incident as a breach of protocol. Output 1 for breach, and 0 for no breach.\n\n### Input:\nUsing a school laptop for personal use\n\n### Response:\n1'}

Index: 99
{'instr